# Análise dos resultados — MLP Dry Bean

Este notebook lê os CSVs gerados por `01_experimentos.ipynb` e produz tabelas Markdown em `results/tables/` para o relatório. Ele não treina modelos.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULTS_CSV = PROJECT_ROOT / "results" / "csv"
RESULTS_TAB = PROJECT_ROOT / "results" / "tables"
RESULTS_TAB.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

def df_to_markdown(df, floatfmt=".4f"):
    """Converte DataFrame em tabela Markdown sem dependências extras."""
    cols = list(df.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |",
             "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, row in df.iterrows():
        vals = []
        for v in row:
            if isinstance(v, (float, np.floating)):
                if np.isnan(v):
                    vals.append("—")
                else:
                    vals.append(format(float(v), floatfmt))
            else:
                vals.append(str(v).replace("|", "\\|"))
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)

def save_table(df, name, floatfmt=".4f"):
    path = RESULTS_TAB / name
    path.write_text(df_to_markdown(df, floatfmt) + "\n", encoding="utf-8")
    print("wrote", path)

def load(name, required=True):
    p = RESULTS_CSV / name
    if not p.exists():
        if required:
            print("AUSENTE:", p, "— execute 01_experimentos.ipynb")
            return None
        return None
    return pd.read_csv(p)

print("Resultados:", RESULTS_CSV)

Resultados: /home/pedro_otavio/uece/ic/segundo-entregavel/results/csv


## Q1 — Inicializações

In [2]:
q1 = load("q1_runs.csv")
if q1 is not None:
    cols = ["run_id", "seed", "epochs", "loss_final", "accuracy_final",
            "loss_min", "loss_min_epoch", "macro_f1_final", "time_s"]
    t = q1[cols].sort_values("seed")
    save_table(t, "q1_tabela.md")
    display(t.round(4))
    print("Média loss final:", q1["loss_final"].mean())
    print("Desvio-padrão loss final:", q1["loss_final"].std())
    print("Média acurácia final:", q1["accuracy_final"].mean())
    print("Desvio-padrão acurácia final:", q1["accuracy_final"].std())

wrote /home/pedro_otavio/uece/ic/segundo-entregavel/results/tables/q1_tabela.md


,run_id,seed,epochs,loss_final,accuracy_final,loss_min,loss_min_epoch,macro_f1_final,time_s
0,q1_s42,42,100,0.1620,0.9436,0.1609,98,0.9527,31.6624
1,q1_s43,43,100,0.1616,0.9437,0.1607,98,0.9532,33.0142
2,q1_s44,44,100,0.1592,0.9424,0.1587,90,0.9523,53.7355
3,q1_s45,45,100,0.1612,0.9429,0.1606,99,0.9523,48.7645
4,q1_s46,46,100,0.1597,0.9431,0.1597,100,0.9518,51.2766


Média loss final: 0.16072878539562224
Desvio-padrão loss final: 0.0012079506136477792
Média acurácia final: 0.9431308150291443
Desvio-padrão acurácia final: 0.0005297690043086578


## Q2 — Learning rate e momento

In [3]:
q2 = load("q2_grid.csv")
if q2 is not None:
    cols = ["run_id", "learning_rate", "momentum", "status", "converged", "epochs",
            "loss_train", "acc_train", "loss_val", "acc_val", "macro_f1_val",
            "weighted_f1_val", "time_s", "selected"]
    t = q2[cols].sort_values(["selected", "macro_f1_val"], ascending=[False, False])
    save_table(t, "q2_tabela.md")
    display(t.round(4))

wrote /home/pedro_otavio/uece/ic/segundo-entregavel/results/tables/q2_tabela.md


,run_id,learning_rate,momentum,status,converged,epochs,loss_train,acc_train,loss_val,acc_val,macro_f1_val,weighted_f1_val,time_s,selected
2,q2_lr0.01_m0.9,0.01,0.9,ok,False,150,0.1524,0.9448,0.1934,0.9334,0.9477,0.9335,90.4728,True
0,q2_lr0.01_m0.5,0.01,0.5,ok,False,150,0.1705,0.9386,0.1997,0.9318,0.9445,0.9317,124.4249,False
1,q2_lr0.01_m0.7,0.01,0.7,ok,False,150,0.1648,0.9398,0.1978,0.9313,0.9442,0.9312,104.5845,False
3,q2_lr0.1_m0.5,0.10,0.5,ok,False,150,0.1459,0.9462,0.1943,0.9286,0.9419,0.9285,105.5240,False
4,q2_lr0.1_m0.7,0.10,0.7,ok,False,150,0.1456,0.9461,0.2081,0.9255,0.9400,0.9255,112.2521,False
6,q2_lr0.5_m0.5,0.50,0.5,ok,False,150,0.1738,0.9402,0.2649,0.9271,0.9394,0.9270,113.9530,False
5,q2_lr0.1_m0.9,0.10,0.9,ok,False,150,0.1656,0.9392,0.2588,0.9239,0.9388,0.9242,103.0194,False
7,q2_lr0.5_m0.7,0.50,0.7,ok,False,150,0.2089,0.9339,0.3099,0.9176,0.9313,0.9174,108.8195,False
8,q2_lr0.5_m0.9,0.50,0.9,ok,False,150,1.6892,0.2988,1.7168,0.2980,0.2027,0.1491,91.7031,False


## Q3 — Topologia

In [4]:
q3 = load("q3_grid.csv")
if q3 is not None:
    cols = ["rank", "run_id", "n_layers", "neurons", "n_params", "status", "converged",
            "epochs", "loss_train", "acc_train", "loss_val", "acc_val",
            "macro_f1_val", "weighted_f1_val", "gap", "time_s", "selected_top4"]
    t = q3[cols].sort_values(["rank", "n_layers", "neurons"], na_position="last")
    save_table(t, "q3_tabela.md")
    display(t.round(4))
    top4 = q3[q3["selected_top4"] == True].sort_values("rank")
    save_table(top4[cols], "q3_top4_tabela.md")

wrote /home/pedro_otavio/uece/ic/segundo-entregavel/results/tables/q3_tabela.md


,rank,run_id,n_layers,neurons,n_params,status,converged,epochs,loss_train,acc_train,loss_val,acc_val,macro_f1_val,weighted_f1_val,gap,time_s,selected_top4
5,1,q3_l1_n60_s42,1,60,1447,ok,False,150,0.1524,0.9448,0.1934,0.9334,0.9477,0.9335,0.0410,154.1231,True
9,2,q3_l1_n100_s42,1,100,2407,ok,False,150,0.1482,0.9465,0.1924,0.9328,0.9473,0.9330,0.0442,143.5863,True
3,3,q3_l1_n40_s42,1,40,967,ok,False,150,0.1564,0.9423,0.1875,0.9323,0.9471,0.9325,0.0310,107.4950,True
8,4,q3_l1_n90_s42,1,90,2167,ok,False,150,0.1511,0.9431,0.1906,0.9323,0.9470,0.9325,0.0395,126.2830,True
1,5,q3_l1_n20_s42,1,20,487,ok,False,150,0.1667,0.9368,0.1978,0.9307,0.9457,0.9308,0.0311,154.0960,False
4,6,q3_l1_n50_s42,1,50,1207,ok,False,150,0.1543,0.9446,0.1892,0.9307,0.9457,0.9308,0.0349,142.0662,False
10,7,q3_l2_n10_s42,2,10,357,ok,False,150,0.1759,0.9358,0.1972,0.9307,0.9447,0.9310,0.0213,116.5399,False
6,8,q3_l1_n70_s42,1,70,1687,ok,False,150,0.1513,0.9433,0.1881,0.9297,0.9446,0.9298,0.0367,126.4709,False
7,9,q3_l1_n80_s42,1,80,1927,ok,False,150,0.1497,0.9437,0.1944,0.9297,0.9436,0.9299,0.0448,136.5083,False
19,10,q3_l2_n100_s42,2,100,12507,ok,False,150,0.1108,0.9577,0.2056,0.9297,0.9435,0.9298,0.0948,218.9292,False


wrote /home/pedro_otavio/uece/ic/segundo-entregavel/results/tables/q3_top4_tabela.md


## Q4 — Quantidade de dados

In [5]:
q4 = load("q4_fractions.csv")
if q4 is not None:
    cols = ["run_id", "fraction", "n_train", "status", "converged", "epochs",
            "loss_train", "acc_train", "loss_val", "acc_val", "macro_f1_val",
            "weighted_f1_val", "time_s", "selected"]
    t = q4[cols].sort_values("fraction")
    save_table(t, "q4_tabela.md")
    display(t.round(4))

wrote /home/pedro_otavio/uece/ic/segundo-entregavel/results/tables/q4_tabela.md


,run_id,fraction,n_train,status,converged,epochs,loss_train,acc_train,loss_val,acc_val,macro_f1_val,weighted_f1_val,time_s,selected
0,q4_frac20,0.2,1524,ok,False,150,0.1521,0.9495,0.2301,0.9307,0.9429,0.9305,20.9326,False
1,q4_frac40,0.4,3048,ok,False,150,0.1556,0.9406,0.2166,0.9255,0.9404,0.9256,28.1193,False
2,q4_frac60,0.6,4572,ok,False,150,0.1506,0.9451,0.2106,0.9297,0.9432,0.9295,34.1574,False
3,q4_frac80,0.8,6096,ok,False,150,0.1543,0.9447,0.2079,0.9297,0.9431,0.9295,45.7460,False
4,q4_frac100,1.0,7621,ok,False,150,0.1524,0.9448,0.1934,0.9334,0.9477,0.9335,51.2518,True


## Q5 — Atributos

In [6]:
q5f = load("q5_features.csv")
if q5f is not None:
    cols = ["feature", "variance", "mutual_info", "all_16", "low_redundancy",
            "top_10_mi", "compact", "no_noise"]
    t = q5f[cols].sort_values("mutual_info", ascending=False)
    save_table(t, "q5_features_tabela.md")
    display(t.round(5))

q5 = load("q5_subsets.csv")
if q5 is not None:
    cols = ["run_id", "subset", "n_features", "status", "reused", "converged",
            "epochs", "loss_train", "acc_train", "loss_val", "acc_val",
            "macro_f1_val", "weighted_f1_val", "time_s", "selected"]
    t = q5[cols].sort_values(["selected", "macro_f1_val"], ascending=[False, False])
    save_table(t, "q5_subsets_tabela.md")
    display(t.round(4))

wrote /home/pedro_otavio/uece/ic/segundo-entregavel/results/tables/q5_features_tabela.md


,feature,variance,mutual_info,all_16,low_redundancy,top_10_mi,compact,no_noise
0,Perimeter,4.540771e+04,1.06276,True,True,True,True,True
1,EquivDiameter,3.459938e+03,1.03731,True,False,True,False,True
2,ConvexArea,8.716746e+08,1.03600,True,False,True,False,True
3,Area,8.457415e+08,1.03600,True,False,True,False,True
4,MajorAxisLength,7.272542e+03,0.99203,True,False,True,False,True
5,ShapeFactor2,0.000000e+00,0.95457,True,True,True,True,True
6,MinorAxisLength,1.996557e+03,0.93145,True,False,True,False,True
7,ShapeFactor1,0.000000e+00,0.92642,True,True,True,True,True
8,ShapeFactor3,9.760000e-03,0.82443,True,True,True,True,True
9,Compactness,3.800000e-03,0.82442,True,False,True,False,True


wrote /home/pedro_otavio/uece/ic/segundo-entregavel/results/tables/q5_subsets_tabela.md


,run_id,subset,n_features,status,reused,converged,epochs,loss_train,acc_train,loss_val,acc_val,macro_f1_val,weighted_f1_val,time_s,selected
1,q5_low_redundancy,low_redundancy,8,ok,False,False,150,0.1522,0.9438,0.1871,0.9307,0.9452,0.9309,50.8499,True
0,q5_all_16,all_16,16,ok,False,False,150,0.1524,0.9448,0.1934,0.9334,0.9477,0.9335,51.3290,False
4,q5_no_noise,no_noise,16,reused,True,False,0,0.1524,0.9448,0.1934,0.9334,0.9477,0.9335,0.0000,False
3,q5_compact,compact,8,ok,False,False,150,0.1522,0.9438,0.1871,0.9307,0.9452,0.9309,54.8343,False
2,q5_top_10_mi,top_10_mi,10,ok,False,False,150,0.2187,0.9171,0.2349,0.9098,0.9235,0.9100,51.7245,False


## Q6 — Validação final no teste

In [7]:
q6 = load("q6_test.csv")
if q6 is not None:
    cols = ["run_id", "n_layers", "neurons", "n_params", "status", "converged",
            "epochs", "loss_train", "acc_train", "macro_f1_train", "loss_test",
            "acc_test", "macro_f1_test", "weighted_f1_test", "precision_macro_test",
            "recall_macro_test", "time_s", "selected"]
    t = q6[cols].sort_values(["selected", "macro_f1_test"], ascending=[False, False])
    save_table(t, "q6_tabela.md")
    display(t.round(4))

q6r = load("q6_classification_report.csv", required=False)
if q6r is not None:
    q6r_md = q6r.reset_index().rename(columns={"index": "class"})
    save_table(q6r_md, "q6_classification_report.md", floatfmt=".4f")
    display(q6r_md.round(4))

wrote /home/pedro_otavio/uece/ic/segundo-entregavel/results/tables/q6_tabela.md


,run_id,n_layers,neurons,n_params,status,converged,epochs,loss_train,acc_train,macro_f1_train,loss_test,acc_test,macro_f1_test,weighted_f1_test,precision_macro_test,recall_macro_test,time_s,selected
1,q6_q3_l1_n100_s42,1,100,1607,ok,False,150,0.1493,0.9448,0.9542,0.1963,0.9280,0.9402,0.9280,0.9416,0.9390,83.8198,True
2,q6_q3_l1_n40_s42,1,40,647,ok,False,150,0.1564,0.9417,0.9516,0.1955,0.9283,0.9401,0.9283,0.9418,0.9387,83.4484,False
3,q6_q3_l1_n90_s42,1,90,1447,ok,False,150,0.1491,0.9446,0.9544,0.1973,0.9275,0.9396,0.9275,0.9413,0.9382,83.6259,False
0,q6_q3_l1_n60_s42,1,60,967,ok,False,150,0.1516,0.9444,0.9543,0.1975,0.9270,0.9395,0.9270,0.9408,0.9384,78.9264,False


wrote /home/pedro_otavio/uece/ic/segundo-entregavel/results/tables/q6_classification_report.md


,class,Unnamed: 0,precision,recall,f1-score,support
0,0,Barbunya,0.9551,0.9118,0.9330,397.000
1,1,Bombay,1.0000,1.0000,1.0000,157.000
2,2,Cali,0.9419,0.9611,0.9514,489.000
3,3,Dermason,0.9143,0.9220,0.9181,1064.000
4,4,Horoz,0.9667,0.9533,0.9599,578.000
5,5,Seker,0.9400,0.9539,0.9469,608.000
6,6,Sira,0.8733,0.8710,0.8722,791.000
7,7,accuracy,0.9280,0.9280,0.9280,0.928
8,8,macro avg,0.9416,0.9390,0.9402,4084.000
9,9,weighted avg,0.9282,0.9280,0.9280,4084.000


## Q7 — Validação cruzada

In [8]:
q7 = load("q7_cv.csv")
if q7 is not None:
    cols = ["fold", "status", "converged", "epochs", "n_train", "n_val",
            "loss_train", "acc_train", "loss_val", "acc_val", "macro_f1_val",
            "weighted_f1_val", "precision_macro_val", "recall_macro_val", "time_s"]
    t = q7[cols].sort_values("fold")
    save_table(t, "q7_folds_tabela.md")
    display(t.round(4))

q7s = load("q7_summary.csv")
if q7s is not None:
    save_table(q7s, "q7_summary_tabela.md", floatfmt=".6f")
    display(q7s.round(6))

wrote /home/pedro_otavio/uece/ic/segundo-entregavel/results/tables/q7_folds_tabela.md


,fold,status,converged,epochs,n_train,n_val,loss_train,acc_train,loss_val,acc_val,macro_f1_val,weighted_f1_val,precision_macro_val,recall_macro_val,time_s
0,0,ok,False,150,7621,1906,0.1521,0.9452,0.1959,0.9334,0.9437,0.9334,0.9462,0.9416,55.5266
1,1,ok,False,150,7621,1906,0.1527,0.9408,0.1910,0.9307,0.9418,0.9311,0.9438,0.9406,56.6814
2,2,ok,False,150,7622,1905,0.1550,0.9421,0.1828,0.9270,0.9406,0.9275,0.9418,0.9403,53.5523
3,3,ok,False,150,7622,1905,0.1469,0.9459,0.1910,0.9354,0.9482,0.9357,0.9491,0.9476,52.2615
4,4,ok,False,150,7622,1905,0.1496,0.9428,0.1804,0.9417,0.9525,0.9417,0.9524,0.9529,54.2360


wrote /home/pedro_otavio/uece/ic/segundo-entregavel/results/tables/q7_summary_tabela.md


,metric,mean,variance,std,min,max
0,loss_train,0.151258,0.000010,0.003083,0.146948,0.154964
1,acc_train,0.943371,0.000005,0.002130,0.940821,0.945946
2,loss_val,0.188222,0.000041,0.006425,0.180397,0.195908
3,acc_val,0.933663,0.000030,0.005495,0.927034,0.941732
4,macro_f1_val,0.945351,0.000024,0.004933,0.940589,0.952475
5,weighted_f1_val,0.933872,0.000028,0.005326,0.927511,0.941709
6,precision_macro_val,0.946660,0.000018,0.004197,0.941825,0.952363
7,recall_macro_val,0.944574,0.000030,0.005515,0.940259,0.952850
8,epochs,150.000000,0.000000,0.000000,150.000000,150.000000
9,time_s,54.451551,2.944813,1.716046,52.261496,56.681360


## Consolidado dos vencedores

In [9]:
w = load("winners.csv")
if w is not None:
    save_table(w, "winners_tabela.md", floatfmt=".5f")
    display(w.round(5))
print("Tabelas geradas em", RESULTS_TAB)

wrote /home/pedro_otavio/uece/ic/segundo-entregavel/results/tables/winners_tabela.md


,stage,run_id,configuration,selection_metric,value,notes
0,Q2,q2_lr0.01_m0.9,lr=0.01; momentum=0.9,macro_f1_val,0.94773,converged=False; epochs=150
1,Q3_top1,q3_l1_n60_s42,layers=1; neurons=60,macro_f1_val,0.94773,loss_val=0.1934; time_s=154.1
2,Q3_top2,q3_l1_n100_s42,layers=1; neurons=100,macro_f1_val,0.94726,loss_val=0.1924; time_s=143.6
3,Q3_top3,q3_l1_n40_s42,layers=1; neurons=40,macro_f1_val,0.94708,loss_val=0.1875; time_s=107.5
4,Q3_top4,q3_l1_n90_s42,layers=1; neurons=90,macro_f1_val,0.94700,loss_val=0.1906; time_s=126.3
5,Q4,q4_frac100,fraction=1.0,macro_f1_val,0.94773,análise de sensibilidade; Q6 usa o desenvolvim...
6,Q5,q5_low_redundancy,subset=low_redundancy; n_features=8,macro_f1_val,0.94518,Perimeter; Extent; Solidity; Roundness; ShapeF...
7,Q6,q6_q3_l1_n100_s42,layers=1; neurons=100; lr=0.01; momentum=0.9; ...,macro_f1_test,0.94022,weighted_f1_test=0.9280
8,Q7,q6_q3_l1_n100_s42,k=5; desenvolvimento 70%,macro_f1_val_mean,0.94535,variance=0.000024; std=0.004933


Tabelas geradas em /home/pedro_otavio/uece/ic/segundo-entregavel/results/tables


## Q2b — Piloto de convergência (experimento separado)

Esta seção lê os resultados do piloto Q2b, que não altera a V1.0.0.

In [10]:
q2b_runs_path = PROJECT_ROOT / "results" / "experiments" / "q2b_500ep" / "csv" / "q2b_runs.csv"
q2b_hist_path = PROJECT_ROOT / "results" / "experiments" / "q2b_500ep" / "csv" / "q2b_history.csv"
if q2b_runs_path.exists():
    q2b_runs = pd.read_csv(q2b_runs_path)
    save_table(q2b_runs, "q2b_tabela.md", floatfmt=".6f")
    display(q2b_runs.round(6))
    q2b_hist = pd.read_csv(q2b_hist_path)
    print("Épocas:", len(q2b_hist))
    print("Melhor loss:", q2b_hist["loss"].min())
    print("Loss final:", q2b_hist["loss"].iloc[-1])
else:
    print("Q2b ainda não executado; rode a seção Q2b em 01_experimentos.ipynb.")

wrote /home/pedro_otavio/uece/ic/segundo-entregavel/results/tables/q2b_tabela.md


,run_id,topology,n_layers,neurons,n_params,optimizer,learning_rate,momentum,batch_size,max_epochs,target_loss,stopped_reason,converged,epochs,loss_final,loss_best,best_epoch,loss_train_eval,acc_train,macro_f1_train,loss_test,acc_test,macro_f1_test,weighted_f1_test,precision_macro_test,recall_macro_test,time_s,seed
0,q2b_3x256_sgd,3x256,3,256,137735,SGD,0.01,0.9,32,500,0.001,target,True,408,0.000967,0.000967,408,0.000943,1.0,1.0,0.786359,0.918462,0.932261,0.918505,0.932507,0.932171,277.791009,42


Épocas: 408
Melhor loss: 0.0009668409475125
Loss final: 0.0009668409475125


## Encerramento

As tabelas acima devem ser copiadas para `report/relatorio.md`, sempre acompanhadas da análise correspondente. Nenhum valor deve ser alterado manualmente; a fonte é `results/csv/`.